# Few-Shot Prompting

Here, we'll continue to focus on out week two example: **prompt engineering**. Since we have our few-shot examples and setlist of skits ready, we'll start by testing few-shot prompting exclusively in this notebook. This means:

* Giving our chosen LLM system instructions *with* ready-made examples
* Providing a test set of user inputs, having it output a humor score, then grading it ourselves
* And recording takeaways that can be used to compare zero-shot prompting

For context, we'll be using Gemini 3 Flash for this step of the project.

In [1]:
# Install the official Google Generative AI SDK ('-q' silences the output, '-U' ensures the most up-to-date version)
!pip install -q -U google-generativeai

In [2]:
from google.colab import userdata
from google import genai
import re # regex: extracting specific number
import io
import csv
import time
from google.genai import types

# Fetch API key
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key = GEMINI_API_KEY)

In [3]:
#upload comedy data
from google.colab import files
uploaded = files.upload()

Saving Copy of DS 340 Final Project Data - Sheet1.csv to Copy of DS 340 Final Project Data - Sheet1.csv


In [4]:
#csv read, extract joke and rating
def fewshot_data_split(uploaded):
  file_name = list(uploaded.keys())[0]
  file_content = uploaded[file_name]
  csv_text = file_content.decode('latin-1')
  csv_reader = csv.reader(io.StringIO(csv_text, newline = ''))
  next(csv_reader)

  testing_sets = []

  for row in csv_reader:
    if not row:
      continue
    try:
      row_id = int(row[0])
      joke = row[1]
      rating = str(row[2])

      # testing sets (15 of each to avoid class imbalance)
      if (1 <= row_id <= 15) or (51 <= row_id <= 65):
        testing_sets.append({
            "id": row_id,
            "joke": joke,
            "rating": rating
            })

    except (ValueError, IndexError):
      continue

  training_sets = [
      # Hard-coded three examples of class 0 & reasons for their classification
      types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: I used to date Hispanic guys, but now I prefer consensual.")]),
      types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is a racist rape joke mocking Hispanic men, making it both overly crude and insensitive. \nScore: 0")]),
      types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: You can get a lot of power out of one inch.")]),
      types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is low and almost too easy of a joke to make, so it's unoriginal and not funny. \nScore: 0")]),
      types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: No one notices the Taylor oil spill because it's a disaster taking place over a long period of time, like Derrick Rose's career.")]),
      types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This comes off as antagonistic since it throws shade at a person, Derrick Rose (whom many consider to be a great but flawed NBA player), and is too unpopular of an opinion, thus not constituting a good joke. \nScore: 0")]),

      # Hard-coded three examples of class 1 & reasons for their classification
      types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: Please, don't call me sir. Call me ma'am.")]),
      types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is funny because it is witty and self-deprecating, given that the user is indeed a man. \nScore: 1")]),
      types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: Hey, I am Conan O'Brien and I'm honored to be the last human host of the Academy Awards. Yes! Yeah! Next year, it's going to be a Waymo in a tux.")]),
      types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is very funny as it's clever and uses observational comedy to comment on the advent of AI in self-driving cars, like Waymo, whilst keeping it wholesome. \nScore: 1")]),
      types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: How did us teaching Diana to drive turn into I'm a male prostitute, you're going to put me out, and you're going to come back in an hour, and you want your trap full?")]),
      types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is funny because, in spite of its unorthodox format (as it doesn't read like a classic joke or pun), it calls out the absurdity of the situation in a unique and unpredictable way. \nScore: 1")])
  ]

  return training_sets, testing_sets

In [8]:
# Passes the data through the LLM using few-shot prompt & returns lists of actual vs. predicted labels
def few_shot_prediction(training_sets, testing_sets):
    y_true = []
    y_pred = []
    # Updated to be binary instead of trinary; added a rubric
    instruction = (
        "You are a professional comedy writer. Evaluate whether the text is funny or not based on this binary rubric:\n"
        "0 (not funny) if there's no comedic intent, missing a punchline, feels forced, too niche, or unoriginal.\n"
        "1 (funny) if it's unpredictable yet clever, witty and unique, intuitive in its wordplay, or isn't overty offensive.\n"
        "First, write one brief sentence explaining your reasoning. Then, on a new line, output the final score in this exact format: 'Score: [0/1]'."
    )

    for test in testing_sets:
        binary_rating = int(test['rating'])
        y_true.append(binary_rating)

        # Create the test input
        test_input = types.Content(
            role = 'user',
            parts = [types.Part.from_text(text = f"Text: {test['joke']}")]
        )

        # Combine examples + test input so the model has context
        full_context = training_sets + [test_input]

        try:
            response = client.models.generate_content(
                model = "gemini-3-flash-preview",
                contents = full_context,
                config = types.GenerateContentConfig(
                    system_instruction = instruction,
                    temperature = 0,
                ),
            )

            # Parsing
            prediction_text = response.text.strip()

            # Find the first digit (0 or 1) in the response
            match = re.search(r'Score:\s*([0-1])', prediction_text)

            if match:
                y_pred.append(int(match.group(1)))
            else:
                # Debug: print out if the model returns something off
                print(f"Warning: Model returned unexpected text for ID {test['id']}: {prediction_text}")
                y_pred.append(0)

            time.sleep(2)

        except Exception as e:
            print(f"Error predicting ID {test['id']}: {e}")
            y_pred.append(0)

    return y_true, y_pred

In [9]:
# accuracy evaluation report
from sklearn.metrics import classification_report

# extract data from the pre-existed dictionary
training_data, testing_data = fewshot_data_split(uploaded)

# run prediction
y_true, y_pred = few_shot_prediction(training_data, testing_data)

# print report
print("\n" + "=" * 50)
print("Few-shot Classification Report")
print("=" * 50)
print(classification_report(
    y_true,
    y_pred,
    labels = [0, 1],
    target_names = ["0 (not funny)", "1 (funny)"],
    zero_division = 0
))



Few-shot Classification Report
               precision    recall  f1-score   support

0 (not funny)       0.89      0.53      0.67        15
    1 (funny)       0.67      0.93      0.78        15

     accuracy                           0.73        30
    macro avg       0.78      0.73      0.72        30
 weighted avg       0.78      0.73      0.72        30

